# Aula 9 — Modelos Discretos I — Apostas Esportivas (simulação em Python)

**Aviso:** este notebook foi elaborado com auxílio de IA (Claude), com fins **exclusivamente educacionais de estatística**. Os números de probabilidade e odds usados são **fictícios**, escolhidos só para ilustrar os três modelos vistos na Aula 9 (uniforme discreta, Bernoulli e geométrica). Não é incentivo a apostas — pelo contrário, o objetivo é mostrar, com matemática, por que apostar repetidamente tende à perda no longo prazo.

Vamos usar o tema de apostas esportivas (bets), bem atual no Brasil, para simular e comparar os valores teóricos que vimos na aula com resultados empíricos (simulados).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)


## 1. Distribuição Uniforme Discreta — a "roleta de bônus" do app

Muitos apps de apostas oferecem uma "roleta de bônus" com resultados igualmente prováveis. Vamos simular uma roleta com 10 resultados possíveis (de 1 a 10 pontos de bônus) e comparar com a teoria vista em aula:

$$E(X) = \frac{n+1}{2}, \qquad Var(X) = \frac{n^2-1}{12}$$

In [ ]:
n_giros = 200_000
k = 10

giros = rng.integers(1, k + 1, size=n_giros)

media_empirica = giros.mean()
var_empirica = giros.var()

media_teorica = (k + 1) / 2
var_teorica = (k**2 - 1) / 12

print(f"E(X) empírico  = {media_empirica:.4f}   |   E(X) teórico  = {media_teorica:.4f}")
print(f"Var(X) empírico = {var_empirica:.4f}   |   Var(X) teórico = {var_teorica:.4f}")


In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(giros, bins=np.arange(0.5, k + 1.5, 1), rwidth=0.8, color="#8A3B3F")
plt.title("Distribuição da roleta de bônus (10 resultados igualmente prováveis)")
plt.xlabel("Resultado")
plt.ylabel("Frequência")
plt.show()


## 2. Distribuição de Bernoulli — uma única aposta

Vamos apostar R\$1,00 em "Time A vence". Suponha que a probabilidade **real** de vitória do Time A seja $p = 0{,}45$, mas a casa oferece uma odd de $2{,}00$ (decimal) para esse resultado.

A odd embute a margem da casa: a probabilidade **implícita** pela odd é $1/2{,}00 = 0{,}50$ — maior que a probabilidade real de $0{,}45$. Essa diferença é o *overround*, a margem que garante lucro à casa no longo prazo.

Seja $X=1$ se o Time A vencer (você ganha $odd-1 = R\$1{,}00$) e $X=0$ se perder (você perde a aposta, $-R\$1{,}00$).

In [ ]:
p_real = 0.45
odd = 2.00
prob_implicita = 1 / odd

n_apostas = 200_000
vitorias = rng.random(n_apostas) < p_real
lucro_por_aposta = np.where(vitorias, odd - 1, -1)

lucro_medio_empirico = lucro_por_aposta.mean()
lucro_esperado_teorico = p_real * (odd - 1) - (1 - p_real) * 1

print(f"Probabilidade real do evento:      {p_real:.2f}")
print(f"Probabilidade implícita pela odd:  {prob_implicita:.2f}")
print(f"Lucro médio por aposta de R$1 (empírico): R$ {lucro_medio_empirico:.4f}")
print(f"E(lucro) teórico:                         R$ {lucro_esperado_teorico:.4f}")


**Leitura do resultado:** o valor esperado do lucro por aposta é **negativo** (cerca de −R\$0,10 a cada R\$1,00 apostado) — mesmo quando o Time A realmente vence quase metade das vezes. A odd foi calibrada pela casa para que isso aconteça sempre que a probabilidade real for menor que a implícita.

## 3. Distribuição Geométrica — quantas apostas até a primeira vitória?

Agora, seja $X$ o número de apostas (independentes, sempre com a mesma probabilidade $p=0{,}45$ de vitória) até sair a **primeira** vitória. Pela aula:

$$E(X) = \frac{1}{p}, \qquad Var(X) = \frac{1-p}{p^2}$$

In [ ]:
n_sequencias = 50_000

# rng.geometric já conta o número de tentativas até o 1º sucesso (mesma definição da aula)
tentativas_ate_vitoria = rng.geometric(p_real, size=n_sequencias)

media_empirica_geo = tentativas_ate_vitoria.mean()
var_empirica_geo = tentativas_ate_vitoria.var()

media_teorica_geo = 1 / p_real
var_teorica_geo = (1 - p_real) / p_real**2

print(f"E(X) empírico  = {media_empirica_geo:.4f}   |   E(X) teórico  = {media_teorica_geo:.4f}")
print(f"Var(X) empírico = {var_empirica_geo:.4f}   |   Var(X) teórico = {var_teorica_geo:.4f}")


In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(tentativas_ate_vitoria, bins=np.arange(0.5, 16.5, 1), rwidth=0.8, color="#8A3B3F")
plt.title("Número de apostas até a primeira vitória (p = 0,45)")
plt.xlabel("Número de tentativas")
plt.ylabel("Frequência")
plt.xlim(0, 15)
plt.show()


## 4. Juntando tudo — o que acontece a longo prazo?

Vamos simular várias sequências de 200 apostas seguidas (sempre R\$1,00, sempre a mesma odd) e acompanhar o lucro acumulado médio ao longo do tempo.

In [ ]:
n_apostas_seq = 200
n_trajetorias = 2_000

resultados = rng.random((n_trajetorias, n_apostas_seq)) < p_real
lucros = np.where(resultados, odd - 1, -1)
lucro_acumulado = np.cumsum(lucros, axis=1)
lucro_acumulado_medio = lucro_acumulado.mean(axis=0)

lucro_esperado_acumulado = lucro_esperado_teorico * np.arange(1, n_apostas_seq + 1)

plt.figure(figsize=(7, 4))
plt.plot(lucro_acumulado_medio, label="Lucro acumulado médio (simulado)", color="#8A3B3F")
plt.plot(lucro_esperado_acumulado, "--", label="Lucro esperado teórico", color="black")
plt.axhline(0, color="gray", linewidth=0.8)
plt.title("Lucro acumulado ao longo de 200 apostas de R$1,00")
plt.xlabel("Número de apostas")
plt.ylabel("Lucro acumulado (R$)")
plt.legend()
plt.show()

print(f"Lucro acumulado médio após {n_apostas_seq} apostas: R$ {lucro_acumulado_medio[-1]:.2f}")
print(f"Lucro esperado teórico após {n_apostas_seq} apostas: R$ {lucro_esperado_acumulado[-1]:.2f}")


**Conclusão:** mesmo com uma probabilidade real de vitória relativamente alta (45%), o lucro acumulado esperado é **sempre negativo e cresce em módulo** conforme o número de apostas aumenta — é a mesma lógica de $E(X)$ que vimos nos três modelos desta aula, só que agora aplicada ao seu próprio bolso. Quanto mais você aposta, mais perto o resultado real fica do valor esperado teórico (Lei dos Grandes Números) — e nesse caso, o valor esperado é negativo por construção da odd.